# Emulating the Linear Matter Power Spectrum

This notebook demonstrates emulating `linear_matter_power(cosmo_params, k, a)`,
which has **two grid variables** (wavenumber k and scale factor a) and returns a
2-D array of shape `(n_a, n_k)`. This exercises the generalized multi-grid
training utilities.

### Steps
1. Compute P(k, a) over a grid of cosmologies using pyccl
2. Flatten the 2-D outputs into scalar training points
3. Train a GP emulator
4. Validate accuracy
5. Hot-swap: replace pyccl with the emulator

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt

from tissage_cosmique.computations.power import linear_matter_power
from tissage_cosmique.emulators import (
    GPEmulator,
    build_training_data,
    params_to_feature_matrix,
    validate_emulator,
)

## 1. Define parameter space and grids

In [ ]:
PARAM_NAMES = ["Omega_c", "h", "sigma8"]
FIXED = dict(Omega_b=0.0486, n_s=0.9667, Omega_k=0.0, w0=-1.0, wa=0.0)

k_grid = np.logspace(-3, 0, 20)   # 20 wavenumbers from 0.001 to 1 Mpc^-1
a_grid = np.array([0.5, 0.8, 1.0])  # 3 scale factors

# Use log10(k) as the feature — k spans 3 orders of magnitude,
# so the GP interpolates much better in log-space
logk_grid = np.log10(k_grid)

rng = np.random.default_rng(42)
n_train, n_test = 50, 10

def make_samples(n, rng):
    return [
        {"Omega_c": rng.uniform(0.22, 0.32), "h": rng.uniform(0.62, 0.75),
         "sigma8": rng.uniform(0.77, 0.87), **FIXED}
        for _ in range(n)
    ]

train_samples = make_samples(n_train, rng)
test_samples = make_samples(n_test, rng)

n_points = n_train * len(a_grid) * len(k_grid)
print(f"Training: {n_train} cosmologies x {len(a_grid)} scale factors x {len(k_grid)} wavenumbers = {n_points} points")
print(f"Feature columns: {PARAM_NAMES + ['log10(k)', 'a']} ({len(PARAM_NAMES) + 2} features)")

## 2. Generate training data

`linear_matter_power(params, k, a)` returns shape `(n_a, n_k)`. We use
`result_axes_order=[1, 0]` since result axis 0 = a (grid index 1) and
axis 1 = k (grid index 0).

**Key trick**: we pass `log10(k)` as the feature grid, but still call
pyccl with the real `k`. Both P(k) and y are in log10-space so the GP
sees smooth, well-scaled features on all axes.

In [ ]:
%%time

# Wrapper: takes log10(k) as feature but calls pyccl with 10**logk
def _lmp_logk(params, logk, a):
    return linear_matter_power(params, 10**logk, a)

X_train, y_train = build_training_data(
    _lmp_logk, train_samples, logk_grid, a_grid,
    param_names=PARAM_NAMES,
    result_axes_order=[1, 0],
)
print(f"X shape: {X_train.shape}  (features: Omega_c, h, sigma8, log10(k), a)")
print(f"y shape: {y_train.shape}")
print(f"P(k) range: [{y_train.min():.1f}, {y_train.max():.1f}] Mpc^3")

## 3. Train a GP emulator

We train in log-space since P(k) spans several orders of magnitude.

In [ ]:
%%time
from sklearn.gaussian_process.kernels import RBF, ConstantKernel

# Train on log10(P(k)) — both features and target in log-space
y_train_log = np.log10(y_train)

# Use an ARD kernel: one length scale per feature dimension.
# This is critical for multi-grid problems where features have
# different characteristic scales.
n_features = X_train.shape[1]
ard_kernel = ConstantKernel() * RBF(length_scale=np.ones(n_features))

emu = GPEmulator(
    feature_names=PARAM_NAMES + ["log10_k", "a"],
    kernel=ard_kernel,
    n_restarts_optimizer=10,
)
emu.fit(X_train, y_train_log)

meta = emu.metadata
print(f"Training R2 (log-space): {meta['training_score']:.6f}")
print(f"Kernel: {meta['kernel']}")

## 4. Validate on held-out cosmologies

In [ ]:
X_test, y_test = build_training_data(
    _lmp_logk, test_samples, logk_grid, a_grid,
    param_names=PARAM_NAMES,
    result_axes_order=[1, 0],
)
y_test_log = np.log10(y_test)

# Validate in log-space
result_log = validate_emulator(emu, X_test, y_test_log)

# Convert to linear-space relative error
y_pred_log = emu.predict(X_test)
y_pred = 10**y_pred_log
rel_err = np.abs(y_pred - y_test) / y_test

print(f"Log-space validation:")
print(f"  R2:   {result_log.r2_score:.6f}")
print(f"  RMSE: {result_log.rmse:.4f} dex")
print(f"")
print(f"Linear-space relative error:")
print(f"  Mean: {np.mean(rel_err):.4%}")
print(f"  Max:  {np.max(rel_err):.4%}")
print(f"  95th: {np.percentile(rel_err, 95):.4%}")

## 5. Visual comparison: P(k) for test cosmologies

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for col, a_val in enumerate(a_grid):
    ax = axes[col]

    for i, params in enumerate(test_samples[:4]):
        pk_truth = linear_matter_power(params, k_grid, np.array([a_val]))[0]

        X_pred = params_to_feature_matrix(
            params, logk_grid, np.array([a_val]),
            param_names=PARAM_NAMES, result_axes_order=[1, 0],
        )
        pk_emu = 10**emu.predict(X_pred)

        color = f"C{i}"
        ax.loglog(k_grid, pk_truth, "-", color=color, lw=1.5,
                  label="pyccl" if i == 0 else None)
        ax.loglog(k_grid, pk_emu, "--", color=color, lw=1,
                  label="GP" if i == 0 else None)

    ax.set_xlabel("k [1/Mpc]")
    ax.set_ylabel("P(k) [Mpc$^3$]")
    ax.set_title(f"a = {a_val}")
    if col == 0:
        ax.legend()

fig.suptitle("Linear matter power spectrum: pyccl (solid) vs GP emulator (dashed)", fontsize=12)
plt.tight_layout()
plt.show()

## 6. Relative error heatmap across (k, a)

In [ ]:
# Evaluate on a finer a grid for one test cosmology
a_fine = np.linspace(0.3, 1.0, 30)
test_p = test_samples[0]

pk_truth = linear_matter_power(test_p, k_grid, a_fine)  # (n_a, n_k)

X_fine = params_to_feature_matrix(
    test_p, logk_grid, a_fine,
    param_names=PARAM_NAMES, result_axes_order=[1, 0],
)
pk_emu = 10**emu.predict(X_fine)
pk_emu_2d = pk_emu.reshape(len(a_fine), len(k_grid))

rel_err_2d = np.abs(pk_emu_2d - pk_truth) / pk_truth * 100

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.pcolormesh(logk_grid, a_fine, rel_err_2d, shading="auto", cmap="RdYlGn_r")
cb = plt.colorbar(im, ax=ax)
cb.set_label("Relative error [%]")
ax.set_xlabel("log10(k [1/Mpc])")
ax.set_ylabel("Scale factor a")
ax.set_title(f"Emulator error map ($\\Omega_c$={test_p['Omega_c']:.3f}, h={test_p['h']:.3f})")
plt.tight_layout()
plt.show()

print(f"Error summary over (k, a) grid:")
print(f"  Mean: {np.mean(rel_err_2d):.3f}%")
print(f"  Max:  {np.max(rel_err_2d):.3f}%")

## 7. Hot-swap: replace pyccl with the emulator

In [ ]:
from tissage_cosmique.swap import swappable, register, original, reset

# Speed comparison: emulator vs pyccl
test_p = test_samples[0]
n_calls = 50

def emu_predict(params, k, a):
    X = params_to_feature_matrix(
        params, np.log10(k), a,
        param_names=PARAM_NAMES, result_axes_order=[1, 0],
    )
    return (10**emu.predict(X)).reshape(len(a), len(k))

t0 = time.time()
for _ in range(n_calls):
    emu_predict(test_p, k_grid, a_grid)
emu_time = time.time() - t0

t0 = time.time()
for _ in range(n_calls):
    linear_matter_power(test_p, k_grid, a_grid)
pyccl_time = time.time() - t0

print(f"Emulator: {n_calls} calls in {emu_time:.3f}s ({emu_time/n_calls*1000:.1f} ms/call)")
print(f"pyccl:    {n_calls} calls in {pyccl_time:.3f}s ({pyccl_time/n_calls*1000:.1f} ms/call)")
print(f"Speedup:  {pyccl_time/emu_time:.1f}x")

## 8. Side-by-side: emulator vs pyccl for the full (k, a) surface

In [ ]:
pk_truth_full = linear_matter_power(test_p, k_grid, a_fine)
pk_emu_full = emu_predict(test_p, k_grid, a_fine)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

vmin = np.log10(pk_truth_full.min())
vmax = np.log10(pk_truth_full.max())

ax = axes[0]
im = ax.pcolormesh(logk_grid, a_fine, np.log10(pk_truth_full), shading="auto", vmin=vmin, vmax=vmax)
ax.set_title("pyccl")
ax.set_xlabel("log10(k)")
ax.set_ylabel("a")

ax = axes[1]
ax.pcolormesh(logk_grid, a_fine, np.log10(pk_emu_full), shading="auto", vmin=vmin, vmax=vmax)
ax.set_title("GP emulator")
ax.set_xlabel("log10(k)")

ax = axes[2]
diff = np.abs(pk_emu_full - pk_truth_full) / pk_truth_full * 100
im2 = ax.pcolormesh(logk_grid, a_fine, diff, shading="auto", cmap="RdYlGn_r")
plt.colorbar(im2, ax=ax, label="Rel. error [%]")
ax.set_title("Relative error")
ax.set_xlabel("log10(k)")

fig.suptitle(f"P(k, a) surface: $\\Omega_c$={test_p['Omega_c']:.3f}, h={test_p['h']:.3f}", fontsize=12)
plt.tight_layout()
plt.show()

## Summary

The power spectrum emulator demonstrates the **multi-grid** workflow:

- `linear_matter_power(params, k, a)` returns a 2-D `(n_a, n_k)` array
- `build_training_data` with `result_axes_order=[1, 0]` flattens this into scalar `(X, y)` pairs with features `[Omega_c, h, sigma8, k, a]`
- Training in log-space handles the wide dynamic range of P(k)
- The GP achieves sub-percent accuracy across the full (k, a) surface

The same pattern generalizes to any N-D computation: pass the grid arrays and specify the axis mapping.